# GuideLLM Performance Benchmark

> **Original source:** [hyogrin/rhoai-lmeval-builder-lab](https://github.com/hyogrin/rhoai-lmeval-builder-lab)  
> **Author:** hyogrin  
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit) for dynamic cluster configuration

In [ ]:
# Auto-configure environment for this cluster
# Env vars are injected via notebook-env ConfigMap (created by deploy.sh)
# Fallback: tries .env file or oc auto-detection for local/standalone use
import os
import subprocess

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)
except ImportError:
    pass

NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
MODEL_NAME = os.environ.get("MODEL_NAME", "")
MODEL_NAMESPACE = os.environ.get("MODEL_NAMESPACE", "")
BASE_URL = os.environ.get("BASE_URL", "")
EVALHUB_URL = os.environ.get("EVALHUB_URL", "http://evalhub.redhat-ods-applications.svc:8080")

# Fallback auto-detection (for running outside a workbench)
if not MODEL_NAME or not BASE_URL:
    detected = ""
    detected_ns = ""
    try:
        result = subprocess.run(
            ["oc", "get", "llminferenceservice", "-A", "--no-headers",
             "-o", "custom-columns=NS:.metadata.namespace,NAME:.metadata.name"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0 and result.stdout.strip():
            parts = result.stdout.strip().split("\n")[0].split()
            detected_ns, detected = parts[0], parts[1]
    except Exception:
        pass

    if detected:
        MODEL_NAME = MODEL_NAME or detected
        MODEL_NAMESPACE = MODEL_NAMESPACE or detected_ns
        if not BASE_URL:
            BASE_URL = f"https://{detected}-kserve-workload-svc.{detected_ns}.svc:8000/v1"

MODEL_NAMESPACE = MODEL_NAMESPACE or NAMESPACE
print(f"Namespace:       {NAMESPACE}")
print(f"Model:           {MODEL_NAME}")
print(f"Model Namespace: {MODEL_NAMESPACE}")
print(f"Base URL:        {BASE_URL}")
print(f"EvalHub:         {EVALHUB_URL}")

if not MODEL_NAME:
    print("\n⚠ MODEL_NAME not set. Either:")
    print("  - Run: scripts/refresh-notebook-env.sh lmeval-demo")
    print("  - Or set MODEL_NAME manually in this cell")

## Step 1: Configuration

In [ ]:
import subprocess

# GuideLLM needs the base /v1 endpoint (not /v1/completions)
GUIDELLM_URL = BASE_URL.replace("/v1/completions", "/v1").replace("/v1/chat/completions", "/v1")

# NOTE: upstream uses ../utils/port_forward.py from hyogrin/rhoai-lmeval-builder-lab
# For in-cluster use, EVALHUB_URL from the config cell above is used directly.
_r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None

print(f"Namespace:         {NAMESPACE}")
print(f"Model Name:        {MODEL_NAME}")
print(f"GuideLLM Endpoint: {GUIDELLM_URL}")
print(f"EvalHub URL:       {EVALHUB_URL}")

## Step 2: Initialize EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(url=GUIDELLM_URL, name=MODEL_NAME)

# Verify GuideLLM provider
providers = client.providers.list()
guidellm_found = any("GuideLLM" in getattr(p, "name", "") for p in providers)
print(f"GuideLLM provider registered: {guidellm_found}")

if guidellm_found:
    for p in providers:
        if "GuideLLM" in getattr(p, "name", ""):
            benchmarks = getattr(p, "benchmarks", [])
            print(f"Available benchmarks ({len(benchmarks)}):")
            for b in benchmarks:
                bid = getattr(b, "benchmark_id", getattr(b, "id", "?"))
                bname = getattr(b, "name", "?")
                print(f"  - {bid}: {bname}")

## Step 3: Quick Performance Baseline

Run a quick performance test to measure single-user latency (TTFT, ITL) with synthetic data.

In [ ]:
from datetime import datetime

ts = datetime.now().strftime("%m%d-%H%M")

quick_request = JobSubmissionRequest(
    name=f"guidellm-quick-{MODEL_NAME}-{ts}",
    description=f"Quick performance baseline for {MODEL_NAME}",
    tags=["performance", "guidellm", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="constant",
            provider_id="guidellm",
            parameters={
                "rate": 2,
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-quick-{MODEL_NAME}-{ts}"),
)

quick_job = client.jobs.submit(quick_request)
print(f"Job submitted: {quick_job.id}")
print(f"State: {quick_job.state.value}")

In [ ]:
import time

job_id = quick_job.id
print(f"Waiting for job {job_id}...")

for i in range(60):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

## Step 4: Rate Sweep Test

Automatically ramps up request rate to discover the saturation point — where latency begins to degrade. This identifies the maximum sustainable throughput.

In [ ]:
sweep_request = JobSubmissionRequest(
    name=f"guidellm-sweep-{MODEL_NAME}-{ts}",
    description=f"Rate sweep to find saturation point for {MODEL_NAME}",
    tags=["performance", "guidellm", "sweep", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="sweep",
            provider_id="guidellm",
            parameters={
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-sweep-{MODEL_NAME}-{ts}"),
)

sweep_job = client.jobs.submit(sweep_request)
print(f"Sweep job submitted: {sweep_job.id}")
print(f"State: {sweep_job.state.value}")

In [ ]:
job_id = sweep_job.id
print(f"Waiting for sweep job {job_id}...")

for i in range(120):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

## Step 5: Constant Load Test

Sustain a fixed request rate to measure steady-state performance. Adjust the `rate` parameter based on your sweep results.

In [ ]:
constant_request = JobSubmissionRequest(
    name=f"guidellm-constant-{MODEL_NAME}-{ts}",
    description=f"Constant load test at 2 req/s for {MODEL_NAME}",
    tags=["performance", "guidellm", "constant", MODEL_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="constant",
            provider_id="guidellm",
            parameters={
                "rate": 2,
                "max_seconds": 120,
                "data": "prompt_tokens=256,output_tokens=128",
                "request_type": "chat_completions",
                "warmup": "5%",
            },
        )
    ],
    experiment=ExperimentConfig(name=f"guidellm-constant-{MODEL_NAME}-{ts}"),
)

constant_job = client.jobs.submit(constant_request)
print(f"Constant load job submitted: {constant_job.id}")
print(f"State: {constant_job.state.value}")

In [ ]:
job_id = constant_job.id
print(f"Waiting for constant load job {job_id}...")

for i in range(120):
    job = client.jobs.get(job_id)
    state = job.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nJob {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(job, "results") and job.results:
    for bm in job.results.benchmarks:
        print(f"\nBenchmark: {bm.id}")
        print(f"MLflow Run: {bm.mlflow_run_id}")
        print(f"Metrics:")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

## Next Steps

- View all performance metrics in the **MLflow UI** (Experiments tab)
- Run **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb** for single Korean MCQ benchmark evaluation
- Run **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** for a unified evaluation (accuracy + performance) tracked in a single MLflow experiment
- Adjust `rate`, `max_seconds`, and `data` parameters based on your deployment scale